<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:12pt;font-weight:800;font-family:Georgia,serif;">
Bronze Layer · AI4I 2020 · Ingestion & Diagnostic
</span><br>
<span style="font-size:9.5pt;font-family:sans-serif;color:#e8e8e8;">
Ingestion brute sans transformation métier. Profilage source. Visualisations raw.
Aucun KPI, aucun renommage définitif, aucune simulation usine.
</span>
</div>

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
00 · Imports & Configuration
</span>
</div>

In [2]:
# ============================================================
# BRONZE LAYER — Imports & Configuration
# Toutes les dépendances et chemins définis une seule fois.
# pathlib garantit la compatibilité Linux/Windows.
# ============================================================

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('bronze')

# ── Chemins ───────────────────────────────────────────────────
ROOT        = Path('..').resolve()
SOURCE_PATH = ROOT / 'data' / 'source' / 'ai4i2020.csv'
BRONZE_PATH = ROOT / 'data' / 'bronze'
OUTPUT_PATH = ROOT / 'outputs' / 'bronze_profile'

BRONZE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# ── Config affichage ──────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Palette Plotly light pro ──────────────────────────────────
LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
    margin=dict(l=50, r=20, t=50, b=40),
)

log.info("Configuration Bronze chargée. ROOT=%s", ROOT)

20:56:56 | INFO | Configuration Bronze chargée. ROOT=C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
01 · Ingestion brute + métadonnées
</span><br>
<span style="font-size:9pt;font-family:sans-serif;color:#e8e8e8;">
Chargement sans transformation. Ajout timestamp d'ingestion et nom de fichier source.
Export bronze CSV.
</span>
</div>

In [3]:
# ============================================================
# INGESTION BRUTE
# On charge le CSV sans modifier aucune valeur.
# On ajoute deux colonnes de métadonnées d'ingestion :
#   ingestion_timestamp : quand le fichier a été ingéré
#   source_file         : d'où vient la donnée
# Ces métadonnées sont la signature de la couche Bronze.
# ============================================================

log.info("Chargement source : %s", SOURCE_PATH)

df_bronze = pd.read_csv(SOURCE_PATH)

# Métadonnées d'ingestion
df_bronze['ingestion_timestamp'] = datetime.utcnow().isoformat()
df_bronze['source_file']         = SOURCE_PATH.name

log.info("Ingestion terminée : %d lignes · %d colonnes", *df_bronze.shape)

# Export Bronze
bronze_file = BRONZE_PATH / 'ai4i_raw.csv'
df_bronze.to_csv(bronze_file, index=False)
log.info("Bronze exporté : %s", bronze_file)

print(f"\n{'='*55}")
print(f"  BRONZE INGESTION SUMMARY")
print(f"{'='*55}")
print(f"  Source        : {SOURCE_PATH.name}")
print(f"  Lignes        : {df_bronze.shape[0]:,}")
print(f"  Colonnes      : {df_bronze.shape[1]}")
print(f"  Ingéré le     : {df_bronze['ingestion_timestamp'].iloc[0]}")
print(f"  Exporté vers  : {bronze_file}")
print(f"{'='*55}\n")

display(df_bronze.drop(columns=['ingestion_timestamp','source_file']).head(10))

20:56:56 | INFO | Chargement source : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\source\ai4i2020.csv
20:56:56 | INFO | Ingestion terminée : 10000 lignes · 16 colonnes
20:56:56 | INFO | Bronze exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\ai4i_raw.csv



  BRONZE INGESTION SUMMARY
  Source        : ai4i2020.csv
  Lignes        : 10,000
  Colonnes      : 16
  Ingéré le     : 2026-04-01T18:56:56.445432
  Exporté vers  : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\ai4i_raw.csv



,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1000,308.6000,1551,42.8000,0,0,0,0,0,0,0
1,2,L47181,L,298.2000,308.7000,1408,46.3000,3,0,0,0,0,0,0
2,3,L47182,L,298.1000,308.5000,1498,49.4000,5,0,0,0,0,0,0
3,4,L47183,L,298.2000,308.6000,1433,39.5000,7,0,0,0,0,0,0
4,5,L47184,L,298.2000,308.7000,1408,40.0000,9,0,0,0,0,0,0
5,6,M14865,M,298.1000,308.6000,1425,41.9000,11,0,0,0,0,0,0
6,7,L47186,L,298.1000,308.6000,1558,42.4000,14,0,0,0,0,0,0
7,8,L47187,L,298.1000,308.6000,1527,40.2000,16,0,0,0,0,0,0
8,9,M14868,M,298.3000,308.7000,1667,28.6000,18,0,0,0,0,0,0
9,10,M14869,M,298.5000,309.0000,1741,28.0000,21,0,0,0,0,0,0


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
02 · Injection contrôlée d'erreurs (PoC Realism)
</span><br>
<span style="font-size:9pt;font-family:sans-serif;color:#e8e8e8;">
Injection de défauts réalistes pour démontrer la valeur de la couche Silver.
Documentée et traçable. Ne modifie pas le fichier source.
</span>
</div>

In [4]:
# ============================================================
# INJECTION CONTRÔLÉE D'ERREURS — PoC Realism
#
# Dans un projet industriel réel, les données brutes arrivent
# rarement propres. On injecte ici des problèmes réalistes
# pour démontrer que la couche Silver sait les détecter
# et les corriger. Chaque injection est documentée.
#
# Types injectés :
#   1. Valeurs manquantes sur colonnes capteurs
#   2. Doublons de lignes
#   3. Valeurs catégorielles mal formées (Type)
#   4. Valeurs numériques hors plage physique
# ============================================================

np.random.seed(99)
df_dirty = df_bronze.copy()

INJECTION_LOG = {}

# 1. Valeurs manquantes — 0.5% des lignes sur les capteurs
N = len(df_dirty)
cols_null = ['Air temperature [K]', 'Process temperature [K]',
             'Rotational speed [rpm]', 'Torque [Nm]']
n_nulls = int(N * 0.005)
for col in cols_null:
    idx = np.random.choice(N, size=n_nulls, replace=False)
    df_dirty.loc[idx, col] = np.nan

INJECTION_LOG['missing_values'] = {
    'colonnes': cols_null,
    'lignes_par_colonne': n_nulls,
    'total_nulls': n_nulls * len(cols_null)
}

# 2. Doublons — 20 lignes dupliquées
dup_idx = np.random.choice(N, size=20, replace=False)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[dup_idx]], ignore_index=True)

INJECTION_LOG['duplicates'] = {'lignes_dupliquees': 20}

# 3. Valeurs catégorielles malformées — Type avec valeurs invalides
bad_type_idx = np.random.choice(len(df_dirty), size=15, replace=False)
df_dirty.loc[bad_type_idx, 'Type'] = np.random.choice(['X', 'low', 'MED', '?'], size=15)

INJECTION_LOG['malformed_categories'] = {
    'colonne': 'Type',
    'valeurs_invalides': ['X', 'low', 'MED', '?'],
    'lignes_affectees': 15
}

# 4. Valeurs hors plage physique — vitesse négative ou > 3000 rpm
bad_speed_idx = np.random.choice(len(df_dirty), size=10, replace=False)
df_dirty.loc[bad_speed_idx, 'Rotational speed [rpm]'] = np.random.choice([-50, 3500, 9999], size=10)

INJECTION_LOG['out_of_range'] = {
    'colonne': 'Rotational speed [rpm]',
    'valeurs_injectees': [-50, 3500, 9999],
    'lignes_affectees': 10
}

# Export dirty bronze
dirty_file = BRONZE_PATH / 'ai4i_raw_dirty.csv'
df_dirty.to_csv(dirty_file, index=False)

# Sauvegarde log injection
inj_file = BRONZE_PATH / 'injection_log.json'
with open(inj_file, 'w') as f:
    json.dump(INJECTION_LOG, f, indent=2)

print(f"\n{'='*55}")
print(f"  INJECTION LOG")
print(f"{'='*55}")
for k, v in INJECTION_LOG.items():
    print(f"  {k:<25} : {v}")
print(f"{'='*55}")
print(f"  Dataset dirty  : {df_dirty.shape[0]:,} lignes · {df_dirty.shape[1]} colonnes")
print(f"  Exporté vers   : {dirty_file}")
print(f"  Log injection  : {inj_file}")
print(f"{'='*55}\n")


  INJECTION LOG
  missing_values            : {'colonnes': ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]'], 'lignes_par_colonne': 50, 'total_nulls': 200}
  duplicates                : {'lignes_dupliquees': 20}
  malformed_categories      : {'colonne': 'Type', 'valeurs_invalides': ['X', 'low', 'MED', '?'], 'lignes_affectees': 15}
  out_of_range              : {'colonne': 'Rotational speed [rpm]', 'valeurs_injectees': [-50, 3500, 9999], 'lignes_affectees': 10}
  Dataset dirty  : 10,020 lignes · 16 colonnes
  Exporté vers   : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\ai4i_raw_dirty.csv
  Log injection  : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\injection_log.json



<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
03 · Profilage source — Structure & Qualité
</span><br>
<span style="font-size:9pt;font-family:sans-serif;color:#e8e8e8;">
Inspection complète du dataset dirty. Ce profil guide les règles de nettoyage Silver.
</span>
</div>

In [5]:
# ============================================================
# PROFILAGE SOURCE — on profile le dataset dirty
# pour simuler ce qu'on verrait en conditions réelles.
# Ce profil est la justification de chaque règle Silver.
# ============================================================

df = df_dirty.copy()

PROFILE = {}

# ── Shape ────────────────────────────────────────────────────
PROFILE['shape'] = {'lignes': df.shape[0], 'colonnes': df.shape[1]}

# ── Types & Nulls ─────────────────────────────────────────────
type_null = pd.DataFrame({
    'Type'              : df.dtypes,
    'Nulls'             : df.isnull().sum(),
    'Nulls_%'           : (df.isnull().sum() / len(df) * 100).round(2),
    'Valeurs_uniques'   : df.nunique()
})
PROFILE['types_nulls'] = type_null.to_dict()

print(f"\n{'='*55}")
print(f"  PROFIL BRONZE — STRUCTURE")
print(f"{'='*55}")
print(f"  Lignes    : {df.shape[0]:,}")
print(f"  Colonnes  : {df.shape[1]}")
print(f"{'='*55}\n")

print("── Types · Nulls · Cardinalité ──")
display(type_null)

# ── Doublons ──────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
PROFILE['duplicates'] = int(n_dupes)

print(f"\n  Doublons détectés : {n_dupes}")
if n_dupes > 0:
    print(f"  ⚠ {n_dupes} lignes dupliquées → à supprimer en Silver")

# ── Stats descriptives colonnes continues ────────────────────
cols_num = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

stats = df[cols_num].describe().T
PROFILE['descriptive_stats'] = stats.to_dict()

print("\n── Statistiques descriptives — variables continues ──")
display(stats)

# ── Distribution catégorielles ────────────────────────────────
print("\n── Distribution Type produit (valeurs attendues : L, M, H) ──")
type_dist = df['Type'].value_counts().to_frame('Occurrences')
type_dist['%'] = (type_dist['Occurrences'] / len(df) * 100).round(2)
display(type_dist)

PROFILE['type_distribution'] = type_dist.to_dict()

# ── Distribution pannes ───────────────────────────────────────
modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Machine failure']
panne_dist = pd.DataFrame({
    'Occurrences' : df[modes].sum(),
    'Taux_%'      : (df[modes].sum() / len(df) * 100).round(3)
})
PROFILE['failure_distribution'] = panne_dist.to_dict()

print("\n── Distribution des modes de panne ──")
display(panne_dist)

# ── Vérification plages physiques ─────────────────────────────
print("\n── Vérification plages physiques ──")

checks = {
    'Air temperature [K]'      : (290, 320),
    'Process temperature [K]'  : (300, 330),
    'Rotational speed [rpm]'   : (0, 3000),
    'Torque [Nm]'              : (0, 80),
    'Tool wear [min]'          : (0, 300),
}

range_issues = {}
for col, (lo, hi) in checks.items():
    out = df[(df[col] < lo) | (df[col] > hi)][col].dropna()
    n_out = len(out)
    range_issues[col] = n_out
    status = "⚠ HORS PLAGE" if n_out > 0 else "✓ OK"
    print(f"  {col:<35} [{lo:>4}, {hi:>4}]  {status}  ({n_out} valeurs)")

PROFILE['range_issues'] = range_issues

# ── Export profil JSON ────────────────────────────────────────
profile_file = BRONZE_PATH / 'bronze_profile.json'
with open(profile_file, 'w') as f:
    json.dump({k: str(v) for k, v in PROFILE.items()}, f, indent=2)

log.info("Profil exporté : %s", profile_file)
print(f"\n  Profil JSON exporté : {profile_file}")


  PROFIL BRONZE — STRUCTURE
  Lignes    : 10,020
  Colonnes  : 16

── Types · Nulls · Cardinalité ──


,Type,Nulls,Nulls_%,Valeurs_uniques
UDI,int64,0,0.0000,10000
Product ID,str,0,0.0000,10000
Type,str,0,0.0000,7
Air temperature [K],float64,50,0.5000,93
Process temperature [K],float64,50,0.5000,82
Rotational speed [rpm],float64,50,0.5000,941
Torque [Nm],float64,50,0.5000,577
Tool wear [min],int64,0,0.0000,246
Machine failure,int64,0,0.0000,2
TWF,int64,0,0.0000,2



  Doublons détectés : 20
  ⚠ 20 lignes dupliquées → à supprimer en Silver

── Statistiques descriptives — variables continues ──


,count,mean,std,min,25%,50%,75%,max
Air temperature [K],9970.0000,300.0026,1.9993,295.3000,298.3000,300.1000,301.5000,304.5000
Process temperature [K],9970.0000,310.0034,1.4837,305.7000,308.8000,310.1000,311.1000,313.8000
Rotational speed [rpm],9970.0000,1543.2821,263.8250,-50.0000,1423.0000,1503.0000,1612.0000,9999.0000
Torque [Nm],9970.0000,39.9813,9.9784,3.8000,33.2000,40.1000,46.7750,76.6000
Tool wear [min],10020.0000,107.9071,63.6396,0.0000,53.0000,108.0000,162.0000,253.0000



── Distribution Type produit (valeurs attendues : L, M, H) ──


,Occurrences,%
Type,,
L,6001,59.8900
M,2999,29.9300
H,1005,10.0300
X,8,0.0800
MED,4,0.0400
?,2,0.0200
low,1,0.0100



── Distribution des modes de panne ──


,Occurrences,Taux_%
TWF,46,0.4590
HDF,115,1.1480
PWF,95,0.9480
OSF,98,0.9780
RNF,19,0.1900
Machine failure,339,3.3830


20:56:57 | INFO | Profil exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\bronze_profile.json



── Vérification plages physiques ──
  Air temperature [K]                 [ 290,  320]  ✓ OK  (0 valeurs)
  Process temperature [K]             [ 300,  330]  ✓ OK  (0 valeurs)
  Rotational speed [rpm]              [   0, 3000]  ⚠ HORS PLAGE  (10 valeurs)
  Torque [Nm]                         [   0,   80]  ✓ OK  (0 valeurs)
  Tool wear [min]                     [   0,  300]  ✓ OK  (0 valeurs)

  Profil JSON exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\bronze\bronze_profile.json


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
04 · Visualisations diagnostiques Bronze
</span><br>
<span style="font-size:9pt;font-family:sans-serif;color:#e8e8e8;">
Analyse visuelle du dataset brut. Distributions, corrélations, anomalies.
Un plot = une question industrielle précise.
</span>
</div>

In [6]:
# ============================================================
# VISUALISATIONS BRONZE — Plotly light pro
# Chaque plot répond à une question industrielle précise.
# hover UI complet, annotations claires, UCL/LCL SPC absents
# (c'est Gold qui les calcule, pas Bronze).
# ============================================================

def save_fig(fig, name):
    """Sauvegarde désactivée — kaleido instable. Export manuel si besoin."""
    pass
    try:
        fig.write_image(str(path))
        log.info("Sauvegardé : %s", path)
    except Exception:
        log.warning("kaleido non installé, PNG non sauvegardé pour %s", name)

# ── Plot 1 : Distributions des capteurs ──────────────────────
print("\n── Plot 1 : Distribution des variables capteurs ──")

fig1 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Température air (K)', 'Température process (K)',
        'Vitesse rotation (rpm)', 'Couple (Nm)',
        'Usure outil (min)', 'Pannes par mode'
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

sensor_cols = [
    ('Air temperature [K]',     1, 1, '#6366f1'),
    ('Process temperature [K]', 1, 2, '#6366f1'),
    ('Rotational speed [rpm]',  1, 3, '#f97316'),
    ('Torque [Nm]',             2, 1, '#f97316'),
    ('Tool wear [min]',         2, 2, '#10b981'),
]

for col, r, c, color in sensor_cols:
    vals = df[col].dropna()
    fig1.add_trace(go.Histogram(
        x=vals, name=col,
        marker_color=color, opacity=0.8,
        showlegend=False,
        hovertemplate=f'<b>{col}</b><br>Valeur : %{{x}}<br>Count : %{{y}}<extra></extra>'
    ), row=r, col=c)

# Pannes par mode
modes_short = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
fig1.add_trace(go.Bar(
    x=modes_short,
    y=[df[m].sum() for m in modes_short],
    marker_color=['#ef4444','#f97316','#eab308','#8b5cf6','#6b7280'],
    showlegend=False,
    hovertemplate='%{x} : %{y} occurrences<extra></extra>'
), row=2, col=3)

fig1.update_layout(
    **LAYOUT,
    title=dict(text='Bronze (Distributions des variables capteurs)',
               font=dict(size=12, color='#111827'), x=0.01),
    height=520
)
fig1.update_xaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
fig1.update_yaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
for ann in fig1.layout.annotations:
    ann.font.update(size=10, color='#6b7280')

fig1.show()
save_fig(fig1, 'plot1_distributions')


── Plot 1 : Distribution des variables capteurs ──


20:56:59 | WARNING | kaleido non installé, PNG non sauvegardé pour plot1_distributions


In [7]:
# ── Plot 2 : Boxplots des capteurs — détection outliers ──────
print("\n── Plot 2 : Boxplots capteurs — détection outliers ──")

fig2 = make_subplots(rows=1, cols=5,
    subplot_titles=['Temp Air', 'Temp Process', 'Vitesse', 'Couple', 'Usure'])

box_cols = [
    ('Air temperature [K]',     1, '#6366f1'),
    ('Process temperature [K]', 2, '#6366f1'),
    ('Rotational speed [rpm]',  3, '#f97316'),
    ('Torque [Nm]',             4, '#f97316'),
    ('Tool wear [min]',         5, '#10b981'),
]

for col, c, color in box_cols:
    fig2.add_trace(go.Box(
        y=df[col].dropna(),
        name=col.split('[')[0].strip(),
        marker_color=color,
        boxmean=True,
        showlegend=False,
        hovertemplate=f'<b>{col}</b><br>%{{y}}<extra></extra>'
    ), row=1, col=c)

fig2.update_layout(
    **LAYOUT,
    title=dict(text='Bronze · Boxplots capteurs (détection outliers)',
               font=dict(size=12, color='#111827'), x=0.01),
    height=420
)
fig2.update_xaxes(showticklabels=False)
fig2.update_yaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
for ann in fig2.layout.annotations:
    ann.font.update(size=10, color='#6b7280')

fig2.show()
save_fig(fig2, 'plot2_boxplots')


── Plot 2 : Boxplots capteurs — détection outliers ──


20:57:00 | WARNING | kaleido non installé, PNG non sauvegardé pour plot2_boxplots


In [8]:
print("\n── Plot 3 : Matrice de corrélation ──")

corr_cols = ['Air temperature [K]', 'Process temperature [K]',
             'Rotational speed [rpm]', 'Torque [Nm]',
             'Tool wear [min]', 'Machine failure']
corr = df[corr_cols].corr().round(3)

labels = ['Temp Air', 'Temp Process', 'Vitesse', 'Couple', 'Usure', 'Panne']

LAYOUT_CORR = {k: v for k, v in LAYOUT.items() if k != 'margin'}

fig3 = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels, y=labels,
    colorscale='RdBu',
    zmid=0, zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    textfont=dict(size=11),
    hovertemplate='%{y} / %{x}<br>Corrélation : %{z:.3f}<extra></extra>',
    colorbar=dict(title='r', tickfont=dict(size=10))
))

fig3.update_layout(
    **LAYOUT_CORR,
    title=dict(text='Bronze : Matrice de corrélation (variables capteurs)',
               font=dict(size=12, color='#111827'), x=0.01),
    height=450,
    margin=dict(l=100, r=20, t=50, b=80),
    xaxis=dict(tickfont=dict(size=10, color='#6b7280')),
    yaxis=dict(tickfont=dict(size=10, color='#6b7280'))
)

fig3.show()
save_fig(fig3, 'plot3_correlation')


── Plot 3 : Matrice de corrélation ──


20:57:00 | WARNING | kaleido non installé, PNG non sauvegardé pour plot3_correlation


In [9]:
import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('bronze')

ROOT        = Path('..').resolve()
SOURCE_PATH = ROOT / 'data' / 'source' / 'ai4i2020.csv'
BRONZE_PATH = ROOT / 'data' / 'bronze'
OUTPUT_PATH = ROOT / 'outputs' / 'bronze_profile'

BRONZE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Pas de margin dans LAYOUT — défini par plot
LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
)

def save_fig(fig, name):
    pass

log.info("Configuration Bronze chargée. ROOT=%s", ROOT)

20:57:00 | INFO | Configuration Bronze chargée. ROOT=C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831


In [15]:
# ── Plot 4 : Usure outil vs Couple — modes de panne ──────────
print("\n── Plot 4 : Usure outil vs Couple — distribution des pannes ──")

mask_ok = df['Machine failure'] == 0

fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=df.loc[mask_ok, 'Tool wear [min]'],
    y=df.loc[mask_ok, 'Torque [Nm]'],
    mode='markers', name='Normal',
    marker=dict(color='#e0e7ff', size=3, opacity=0.4),
    hovertemplate='Usure : %{x} min<br>Couple : %{y} Nm<extra></extra>'
))

panne_colors = {
    'TWF': ('#ef4444', 'Usure outil'),
    'HDF': ('#f97316', 'Thermique'),
    'PWF': ('#eab308', 'Puissance'),
    'OSF': ('#8b5cf6', 'Surcharge'),
    'RNF': ('#6b7280', 'Aleatoire'),
}

for mode, (color, label) in panne_colors.items():
    mask = df[mode] == 1
    if mask.sum() == 0:
        continue
    fig4.add_trace(go.Scatter(
        x=df.loc[mask, 'Tool wear [min]'],
        y=df.loc[mask, 'Torque [Nm]'],
        mode='markers', name=label,
        marker=dict(color=color, size=6, opacity=0.9,
                    line=dict(width=0.5, color='#fff')),
        hovertemplate=f'<b>{label}</b><br>Usure : %{{x}} min<br>Couple : %{{y}} Nm<extra></extra>'
    ))

fig4.update_layout(
    **LAYOUT,
    title=dict(text='Bronze · Usure outil vs Couple (Distribution des modes de panne)',
               font=dict(size=13, color='#111827'), x=0.01),
    xaxis=dict(title='Usure outil (min)', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af')),
    yaxis=dict(title='Couple (Nm)', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af')),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), x=1.01, y=1, xanchor='left'),
    height=440,
    margin=dict(l=50, r=140, t=50, b=40),
    annotations=[dict(
        text='Les pannes TWF se concentrent au-delà de 200 min d\'usure',
        xref='paper', yref='paper', x=0.01, y=-0.12,
        showarrow=False, font=dict(size=9, color='#6b7280'), align='left'
    )]
)

fig4.show()
save_fig(fig4, 'plot4_wear_torque')


── Plot 4 : Usure outil vs Couple — distribution des pannes ──


In [11]:
print("\n── Plot 5 : Vitesse vs Couple : Zone de danger mécanique ──")
mask_ok    = df['Machine failure'] == 0
mask_panne = df['Machine failure'] == 1
fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=df.loc[mask_ok, 'Rotational speed [rpm]'],
    y=df.loc[mask_ok, 'Torque [Nm]'],
    mode='markers', name='Normal',
    marker=dict(color='#e0e7ff', size=3, opacity=0.4),
    hovertemplate='Vitesse : %{x} rpm<br>Couple : %{y} Nm<extra></extra>'
))

mask_panne = df['Machine failure'] == 1
fig5.add_trace(go.Scatter(
    x=df.loc[mask_panne, 'Rotational speed [rpm]'],
    y=df.loc[mask_panne, 'Torque [Nm]'],
    mode='markers', name='Panne',
    marker=dict(color='#ef4444', size=5, opacity=0.85,
                line=dict(width=0.5, color='#fff')),
    hovertemplate='Vitesse : %{x} rpm<br>Couple : %{y} Nm<br><b>Panne</b><extra></extra>'
))

rpm_r = np.linspace(800, 4000, 300)
t_lo  = (3500 * 9550) / (rpm_r * 1000)
t_hi  = (9000 * 9550) / (rpm_r * 1000)

fig5.add_trace(go.Scatter(
    x=np.concatenate([rpm_r, rpm_r[::-1]]),
    y=np.concatenate([t_hi, t_lo[::-1]]),
    fill='toself', fillcolor='rgba(99,102,241,0.07)',
    line=dict(color='#6366f1', width=1, dash='dot'),
    name='Zone nominale (3 500 à 9 000 W)',
    hoverinfo='skip'
))

fig5.update_layout(
    **LAYOUT,
    title=dict(text='Bronze · Vitesse vs Couple (zone de danger mécanique)',
               font=dict(size=12, color='#111827'), x=0.01),
    xaxis=dict(title='Vitesse rotation (rpm)', range=[800, 4000],
               gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af')),
    yaxis=dict(title='Couple (Nm)', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af')),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    height=440,
    margin=dict(l=50, r=20, t=60, b=40)
)

fig5.show()
save_fig(fig5, 'plot5_speed_torque')


── Plot 5 : Vitesse vs Couple : Zone de danger mécanique ──


In [12]:
# ── Plot 6 : Delta thermique — signal précurseur HDF ─────────
print("\n── Plot 6 : Delta thermique (Process − Air) et pannes HDF ──")

df_plot = df.copy()
df_plot['delta_T'] = df_plot['Process temperature [K]'] - df_plot['Air temperature [K]']
df_plot['idx']     = range(len(df_plot))
df_plot['delta_T_ma'] = df_plot['delta_T'].rolling(window=100, min_periods=1).mean()

fig6 = go.Figure()

fig6.add_trace(go.Scatter(
    x=df_plot['idx'], y=df_plot['delta_T'],
    mode='lines', name='Delta thermique (K)',
    line=dict(color='#c7d2fe', width=0.8), opacity=0.7,
    hovertemplate='Session : %{x}<br>Delta T : %{y:.2f} K<extra></extra>'
))

fig6.add_trace(go.Scatter(
    x=df_plot['idx'], y=df_plot['delta_T_ma'],
    mode='lines', name='Moyenne mobile 100 sessions',
    line=dict(color='#6366f1', width=2.2),
    hovertemplate='Session : %{x}<br>Moy Delta T : %{y:.2f} K<extra></extra>'
))

mask_hdf = df_plot['HDF'] == 1
fig6.add_trace(go.Scatter(
    x=df_plot.loc[mask_hdf, 'idx'],
    y=df_plot.loc[mask_hdf, 'delta_T'],
    mode='markers', name='Panne thermique (HDF)',
    marker=dict(
        color='rgba(239, 68, 68, 0.55)',
        size=4,
        symbol='circle',
        line=dict(width=1, color='rgba(239, 68, 68, 0.9)')
    ),
    hovertemplate='Session : %{x}<br>Delta T : %{y:.2f} K<br><b>HDF</b><extra></extra>'
))

seuil_hdf = df_plot.loc[mask_hdf, 'delta_T'].min() if mask_hdf.sum() > 0 else 8.6
fig6.add_hline(
    y=seuil_hdf,
    line_dash='dot',
    line_color='#ef4444',
    line_width=1.2,
    annotation_text=f'Seuil critique HDF ≈ {seuil_hdf:.1f} K',
    annotation_font_size=9,
    annotation_font_color='#ef4444',
    annotation_position='top right'
)

y_min = min(df_plot['delta_T'].min(), seuil_hdf) - 0.3
y_max = df_plot['delta_T'].max() + 0.3

fig6.update_layout(
    **LAYOUT,
    autosize=True,
    title=dict(
        text='Bronze · Delta thermique et pannes HDF (Signal précurseur)',
        font=dict(size=12, color='#111827'),
        x=0.01
    ),
    xaxis=dict(
        title='Session (index)',
        gridcolor='#f3f4f6',
        linecolor='#e5e7eb',
        tickfont=dict(size=10, color='#9ca3af'),
        range=[df_plot['idx'].max(), df_plot['idx'].min()]   # reversed + full width
    ),
    yaxis=dict(
        title='Delta T (K)',
        gridcolor='#f3f4f6',
        linecolor='#e5e7eb',
        tickfont=dict(size=10, color='#9ca3af'),
        range=[y_min, y_max]   # tighter vertical fit
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=10),
        orientation='h',
        y=1.08
    ),
    margin=dict(l=45, r=20, t=60, b=45),   # less empty outer space
    height=420
)

fig6.show()
save_fig(fig6, 'plot6_delta_thermique')


── Plot 6 : Delta thermique (Process − Air) et pannes HDF ──


In [13]:
# ── Plot 7 : Impact des modes de panne ───────────────────────
print("\n── Plot 7 : Impact des modes de panne — Occurrences et durées ──")

modes_labels = ['Usure outil', 'Thermique', 'Puissance', 'Surcharge', 'Aléatoire']
modes_cols   = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
occurrences  = [int(df[m].sum()) for m in modes_cols]
durees_moy   = [120, 95, 60, 80, 30]
couleurs     = ['#ef4444','#f97316','#eab308','#8b5cf6','#6b7280']

fig7 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Occurrences par mode', 'Durée arrêt estimée (min)'],
    horizontal_spacing=0.14
)

for i, (label, occ, duree, color) in enumerate(
        zip(modes_labels, occurrences, durees_moy, couleurs)):
    fig7.add_trace(go.Bar(
        y=[label], x=[occ], orientation='h',
        marker_color=color, opacity=0.85,
        text=[str(occ)], textposition='outside',
        textfont=dict(size=10),
        name=label, showlegend=False,
        hovertemplate=f'<b>{label}</b><br>Occurrences : {occ}<extra></extra>'
    ), row=1, col=1)

    fig7.add_trace(go.Bar(
        y=[label], x=[duree], orientation='h',
        marker_color=color, opacity=0.85,
        text=[f'{duree} min'], textposition='outside',
        textfont=dict(size=10),
        name=label, showlegend=False,
        hovertemplate=f'<b>{label}</b><br>Durée : {duree} min<extra></extra>'
    ), row=1, col=2)

fig7.update_layout(
    **LAYOUT,
    title=dict(text='Bronze · Impact des modes de panne (Occurrences et durées d\'arrêt)',
               font=dict(size=12, color='#111827'), x=0.01),
    height=380,
    margin=dict(l=110, r=60, t=50, b=50),
    barmode='group'
)
fig7.update_xaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
fig7.update_yaxes(gridcolor='#f3f4f6', linecolor='#e5e7eb',
                  tickfont=dict(size=9, color='#9ca3af'))
for ann in fig7.layout.annotations:
    ann.font.update(size=10, color='#6b7280')

fig7.show()
save_fig(fig7, 'plot7_failure_impact')


── Plot 7 : Impact des modes de panne — Occurrences et durées ──


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
05 · Bilan Bronze -> décisions Silver
</span>
</div>

In [14]:
# ============================================================
# BILAN BRONZE
# Ce print est le pont entre Bronze et Silver.
# Chaque ligne est une décision de nettoyage documentée.
# ============================================================

print(f"\n{'='*60}")
print(f"  BILAN BRONZE — DÉCISIONS POUR LA COUCHE SILVER")
print(f"{'='*60}")

decisions = [
    ("Valeurs manquantes",   f"{df.isnull().sum().sum()} nulls détectés",
     "Imputation médiane sur les capteurs numériques"),
    ("Doublons",             f"{df.duplicated().sum()} lignes dupliquées",
     "Suppression des doublons stricts"),
    ("Type produit malformé",f"Valeurs invalides détectées : {df[~df['Type'].isin(['L','M','H'])]['Type'].unique().tolist()}",
     "Suppression des lignes avec Type invalide"),
    ("Vitesse hors plage",   f"{(df['Rotational speed [rpm]'] < 0).sum() + (df['Rotational speed [rpm]'] > 3000).sum()} valeurs hors [0, 3000]",
     "Clipping ou suppression selon la politique Silver"),
    ("Températures Kelvin",  "Présentes — unité non industrielle",
     "Conversion en Celsius dans Silver"),
    ("Colonnes à renommer",  "Noms longs avec crochets et unités",
     "Renommage standardisé snake_case dans Silver"),
    ("Signal thermique HDF", f"Seuil critique ≈ {seuil_hdf:.1f} K — signal précurseur validé",
     "Conserver delta_T comme feature clé dans Gold"),
    ("Usure outil TWF",      "Pannes TWF concentrées > 200 min — validé visuellement",
     "Modèle quadratique Usine A justifié"),
]

for sujet, constat, decision in decisions:
    print(f"\n  {'─'*56}")
    print(f"  ► {sujet}")
    print(f"    Constat  : {constat}")
    print(f"    Décision : {decision}")

print(f"\n{'='*60}")
print(f"  OUTPUTS BRONZE")
print(f"{'='*60}")
print(f"  ✓ {BRONZE_PATH / 'ai4i_raw.csv'}")
print(f"  ✓ {BRONZE_PATH / 'ai4i_raw_dirty.csv'}")
print(f"  ✓ {BRONZE_PATH / 'bronze_profile.json'}")
print(f"  ✓ {BRONZE_PATH / 'injection_log.json'}")
print(f"  ✓ {OUTPUT_PATH} — 7 plots PNG")
print(f"{'='*60}\n")


  BILAN BRONZE — DÉCISIONS POUR LA COUCHE SILVER

  ────────────────────────────────────────────────────────
  ► Valeurs manquantes
    Constat  : 200 nulls détectés
    Décision : Imputation médiane sur les capteurs numériques

  ────────────────────────────────────────────────────────
  ► Doublons
    Constat  : 20 lignes dupliquées
    Décision : Suppression des doublons stricts

  ────────────────────────────────────────────────────────
  ► Type produit malformé
    Constat  : Valeurs invalides détectées : ['MED', '?', 'low', 'X']
    Décision : Suppression des lignes avec Type invalide

  ────────────────────────────────────────────────────────
  ► Vitesse hors plage
    Constat  : 10 valeurs hors [0, 3000]
    Décision : Clipping ou suppression selon la politique Silver

  ────────────────────────────────────────────────────────
  ► Températures Kelvin
    Constat  : Présentes — unité non industrielle
    Décision : Conversion en Celsius dans Silver

  ──────────────────────────